In [ ]:
import os
import librosa
import kaggle
import numpy as np
from glob import glob
from pathlib import Path
import pandas as pd
import math

from matplotlib import pyplot as plt

download recordings

In [ ]:
if not os.path.exists("./data"):
    kaggle.api.authenticate()
    kaggle.api.dataset_download_files('quochoangvuvan/ml-voice-recognition', path=".", unzip=True)
    print(kaggle.api.dataset_list_files('quochoangvuvan/ml-voice-recognition').files)
    kaggle.api.dataset_metadata('quochoangvuvan/ml-voice-recognition', path=".")

#class0_files = glob("./data/Class0/*/*.mp3")
#class1_files = glob("./data/Class1/*/*.mp3")

class0_dirs = [d for d in Path("./data/Class0").iterdir()]
class0_speakers = {}
for speaker_dir in class0_dirs:
    speaker_name = speaker_dir.name
    audio_files = list(speaker_dir.glob('*.mp3'))
    class0_speakers[speaker_name] = audio_files

class1_dirs = [d for d in Path("./data/Class1").iterdir()]
class1_speakers = {}
for speaker_dir in class0_dirs:
    speaker_name = speaker_dir.name
    audio_files = list(speaker_dir.glob('*.mp3'))
    class1_speakers[speaker_name] = audio_files


    #todo jak z noise

In [ ]:
#todo odchylenia standardowe per klasa??????? no chyba impossible
#todo co z mean????????? std??????????? max????????? min????????? -> jest na dole

In [ ]:
recording_lengths_class0 = []
recording_lengths_class1 = []

speaker_lengths_class0 = {}
speaker_lengths_class1 = {}

for speaker, files in class0_speakers.items():
    total = 0
    for f in files:
        y, sr = librosa.load(f, sr=None)
        length = len(y) / sr
        recording_lengths_class0.append(length)
        total += length
    speaker_lengths_class0[speaker] = total

for speaker, files in class1_speakers.items():
    total = 0
    for f in files:
        y, sr = librosa.load(f, sr=None)
        length = len(y) / sr
        recording_lengths_class1.append(length)
        total += length
    speaker_lengths_class1[speaker] = total

In [ ]:
# cumulative number of recordings in class1, in class0  barplot
counts = [len(recording_lengths_class0), len(recording_lengths_class1)]

plt.bar(["Class0", "Class1"], counts, color=["red", "blue"])
plt.title("Cumulative number of recordings")
plt.ylabel("Count")
plt.show()

In [ ]:
# cumulative length of recordings in class1, in class0  barplot
total_len_class0 = sum(recording_lengths_class0)
total_len_class1 = sum(recording_lengths_class1)

plt.bar(["Class0", "Class1"], [total_len_class0, total_len_class1], color=["red", "blue"])
plt.title("Total duration of recordings")
plt.ylabel("Duration (seconds)")
plt.show()

In [ ]:
#TODO cumulative length of recordings in class1, speakers in class0 and noise in class 0 type barplot
plt.figure()

plt.bar(speaker_lengths_class0.keys(),
        list(speaker_lengths_class0.values()),
        color="red",
        label="Class0")

plt.bar(speaker_lengths_class1.keys(),
        list(speaker_lengths_class1.values()),
        color="blue",
        label="Class1")

plt.title("Cumulative durations of recordings per speaker")
plt.ylabel("Seconds")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#TODO cumulative length of recordings per speaker in class1, per speaker in class0 and per noise type(or whole noise again? or without it at all) in class 0 type histogram

plt.figure()
plt.hist(list(speaker_lengths_class0.values()), bins=20, alpha=0.6, color='red', label="Class0")
plt.hist(list(speaker_lengths_class1.values()), bins=20, alpha=0.6, color='blue', label="Class1")
plt.title("Cumulative durations of recordings for each speaker")
plt.xlabel("Seconds")
plt.ylabel("Count")
plt.legend()
plt.show()

In [ ]:
#TODO cumulative length of recordings per speaker in class1, per speaker in class0 and per noise type(or whole noise again? or without it at all) in class 0 type boxplot
plt.figure()
plt.boxplot([list(speaker_lengths_class0.values()),list(speaker_lengths_class1.values())], labels=["Class0", "Class1"])
plt.title("Cumulative durations of recordings for each speaker")
plt.ylabel("Seconds")
plt.show()

In [ ]:
#TODO length of a single recording in class1, single recording of a speaker in class0 and single recording of noise in class 0 type histogram
plt.figure()
plt.hist(recording_lengths_class0, bins=30, alpha=0.6, color='red', label="Class0 recordings")
plt.hist(recording_lengths_class1, bins=30, alpha=0.6, color='blue', label="Class1 recordings")
plt.title("Single recording duration")
plt.xlabel("Seconds")
plt.ylabel("Count")
plt.legend()
plt.show()

In [ ]:
#TODO length of a single recording in class1, single recording of a speaker in class0 and single recording of noise in class 0 type boxplots
plt.figure()
plt.boxplot([recording_lengths_class0, recording_lengths_class1], labels=["Class0", "Class1"])
plt.title("Single recording durations")
plt.ylabel("Seconds")
plt.show()

In [ ]:
#TODO length of a single recording in class1, and single recording in class 0 type histogram

In [ ]:
#TODO length of a single recording in class1, and single recording in class 0 type boxplot

In [ ]:
class0_count = len(class0_speakers)
class1_count = len(class1_speakers)
fig, ax = plt.subplots()
x_positions = [0, 1]
bars = ax.bar(['class0', 'class1'], [class0_count, class1_count])
#todo tutaj noise jako 1 speaker rozrozni
ax.bar_label(bars, label_type='edge', color='blue')
ax.title.set_text('Number of speakers in classes')
plt.show()

In [ ]:
import gc

fig, ax = plt.subplots()
current_position = 1

for speaker in class0_speakers:
    speaker_arr = np.concatenate([librosa.load(audio_file, sr=None)[0] for audio_file in class0_speakers[speaker]])
    ax.boxplot(speaker_arr, label=f"Class0_{speaker}", positions=[current_position], patch_artist=True,
                boxprops=dict(facecolor="red"), medianprops=dict(color='green'))
    del speaker_arr; gc.collect()
    current_position += 1


for speaker in class1_speakers:
    speaker_arr = np.concatenate([librosa.load(audio_file, sr=None)[0] for audio_file in class1_speakers[speaker]])
    ax.boxplot(speaker_arr, label=f"Class1_{speaker}", positions=[current_position], patch_artist=True,
                boxprops=dict(facecolor="blue"),medianprops=dict(color='green'))
    del speaker_arr; gc.collect()
    current_position += 1

ax.set_title('Amplitude Distributions for All Recordings per Speaker')
ax.set_ylabel('Amplitude')
plt.show()

In [ ]:
n_speakers = len(class0_count+class1_count)
n_cols = min(5, n_speakers)
n_rows = math.ceil(n_speakers / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 3*n_rows))
axes = axes.flatten()
current_position = 1

speaker_class0_stats = {}
speaker_class1_stats = {}
recording_class0_stats = {}
recording_class1_stats = {}

for speaker in class0_speakers:
    arrays = []
    for audio_file in class0_speakers[speaker]:
        y, sr = librosa.load(audio_file, sr=None)
        arrays.append(y)

        recording_class0_stats[audio_file] = {
            "mean": (np.mean(y)),
            "std": (np.std(y)),
            "min": (np.min(y)),
            "max": (np.max(y))
        }

    speaker_arr = np.concatenate(arrays)

    axes[current_position-1].hist(speaker_arr, bins=100, density=True, color="red")
    axes[current_position-1].set_title(f"Class0_{current_position}_{speaker} amplitude density")
    axes[current_position-1].set_xlabel("Amplitude")
    axes[current_position-1].set_ylabel("Density")

    speaker_class0_stats[f"{speaker}"] = {
        "mean": (np.mean(speaker_arr)),
        "std": (np.std(speaker_arr)),
        "min": (np.min(speaker_arr)),
        "max": (np.max(speaker_arr))
    }

    del speaker_arr, arrays
    gc.collect()

    current_position += 1

for speaker in class1_speakers:
    arrays = []
    for audio_file in class1_speakers[speaker]:
        y, sr = librosa.load(audio_file, sr=None)
        arrays.append(y)

        recording_class1_stats[audio_file] = {
            "mean": (np.mean(y)),
            "std": (np.std(y)),
            "min": (np.min(y)),
            "max": (np.max(np.abs(y)))
        }

    speaker_arr = np.concatenate(arrays)

    axes[current_position-1].hist(speaker_arr, bins=100, density=True, color="blue")
    axes[current_position-1].set_title(f"Class1_{current_position}_{speaker} amplitude density")
    axes[current_position-1].set_xlabel("Amplitude")
    axes[current_position-1].set_ylabel("Density")

    speaker_class1_stats[f"Class1_{speaker}"] = {
        "mean":(np.mean(speaker_arr)),
        "std": (np.std(speaker_arr)),
        "min": (np.min(speaker_arr)),
        "max": (np.max(speaker_arr))
    }

    del speaker_arr, arrays
    gc.collect()

    current_position += 1

#hide unused subplots
for i in range(current_position-1, len(axes)):
    axes[i].axis('off')

fig.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10,4))
plt.hist(speaker_class0_stats["mean"], bins=20, alpha=0.6, color='red', label='Class0')
plt.hist(speaker_class0_stats["mean"], bins=20, alpha=0.6, color='blue', label='Class1')
plt.title("Mean amplitude per speaker")
plt.xlabel("Mean")
plt.ylabel("Count")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10,4))
plt.hist(speaker_class0_stats["std"], bins=20, alpha=0.6, color='red', label='Class0')
plt.hist(speaker_class0_stats["std"], bins=20, alpha=0.6, color='blue', label='Class1')
plt.title("Std amplitude per speaker")
plt.xlabel("Std")
plt.ylabel("Count")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10,4))
plt.hist(speaker_class0_stats["min"], bins=20, alpha=0.6, color='red', label='Class0')
plt.hist(speaker_class0_stats["min"], bins=20, alpha=0.6, color='blue', label='Class1')
plt.title("Min amplitude per speaker")
plt.xlabel("Min")
plt.ylabel("Count")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10,4))
plt.hist(speaker_class0_stats["max"], bins=20, alpha=0.6, color='red', label='Class0')
plt.hist(speaker_class0_stats["max"], bins=20, alpha=0.6, color='blue', label='Class1')
plt.title("Max amplitude per speaker")
plt.xlabel("Max")
plt.ylabel("Count")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10,4))
plt.hist([v["mean"] for v in recording_class0_stats.values()], bins=30, alpha=0.6, color='red', label='Class0')
plt.hist([v["mean"] for v in recording_class1_stats.values()], bins=30, alpha=0.6, color='blue', label='Class1')
plt.title("Mean amplitude per recording")
plt.xlabel("Mean")
plt.ylabel("Count")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10,4))
plt.hist([v["std"] for v in recording_class0_stats.values()], bins=30, alpha=0.6, color='red', label='Class0')
plt.hist([v["std"] for v in recording_class1_stats.values()], bins=30, alpha=0.6, color='blue', label='Class1')
plt.title("Std amplitude per recording")
plt.xlabel("Std")
plt.ylabel("Count")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10,4))
plt.hist([v["max"] for v in recording_class0_stats.values()], bins=30, alpha=0.6, color='red', label='Class0')
plt.hist([v["max"] for v in recording_class1_stats.values()], bins=30, alpha=0.6, color='blue', label='Class1')
plt.title("Max amplitude per recording")
plt.xlabel("Max")
plt.ylabel("Count")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10,4))
plt.hist([v["min"] for v in recording_class0_stats.values()], bins=30, alpha=0.6, color='red', label='Class0')
plt.hist([v["min"] for v in recording_class1_stats.values()], bins=30, alpha=0.6, color='blue', label='Class1')
plt.title("Min amplitude per recording")
plt.xlabel("Min")
plt.ylabel("Count")
plt.legend()
plt.show()